# **Space X  Falcon 9 First Stage Landing Prediction**
## Web scraping Falcon 9 and Falcon Heavy Launches Records from Wikipedia
In this lab, you will be performing web scraping to collect Falcon 9 historical launch records from a Wikipedia page titled `List of Falcon 9 and Falcon Heavy launches`

https://en.wikipedia.org/wiki/List_of_Falcon_9_and_Falcon_Heavy_launches


![](https://cf-courses-data.s3.us.cloud-object-storage.appdomain.cloud/IBM-DS0321EN-SkillsNetwork/labs/module_1_L2/images/Falcon9_rocket_family.svg)
![](https://cf-courses-data.s3.us.cloud-object-storage.appdomain.cloud/IBM-DS0321EN-SkillsNetwork/labs/module_1_L2/images/falcon9-launches-wiki.png)


In [9]:
import sys

import requests
from bs4 import BeautifulSoup
import re
import unicodedata
import pandas as pd

In [11]:
def date_time(table_cells):
    """
    This function returns the data and time from the HTML  table cell
    Input: the  element of a table data cell extracts extra row
    """
    return [data_time.strip() for data_time in list(table_cells.strings)][0:2]

def booster_version(table_cells):
    """
    This function returns the booster version from the HTML  table cell 
    Input: the  element of a table data cell extracts extra row
    """
    out=''.join([booster_version for i,booster_version in enumerate( table_cells.strings) if i%2==0][0:-1])
    return out

def landing_status(table_cells):
    """
    This function returns the landing status from the HTML table cell 
    Input: the  element of a table data cell extracts extra row
    """
    out=[i for i in table_cells.strings][0]
    return out


def get_mass(table_cells):
    mass=unicodedata.normalize("NFKD", table_cells.text).strip()
    if mass:
        mass.find("kg")
        new_mass=mass[0:mass.find("kg")+2]
    else:
        new_mass=0
    return new_mass


def extract_column_from_header(row):
    """
    This function returns the landing status from the HTML table cell 
    Input: the  element of a table data cell extracts extra row
    """
    if (row.br):
        row.br.extract()
    if row.a:
        row.a.extract()
    if row.sup:
        row.sup.extract()
        
    colunm_name = ' '.join(row.contents)
    
    # Filter the digit and empty names
    if not(colunm_name.strip().isdigit()):
        colunm_name = colunm_name.strip()
        return colunm_name    


In [22]:
static_url = "https://en.wikipedia.org/w/index.php?title=List_of_Falcon_9_and_Falcon_Heavy_launches&oldid=1027686922"
# Send a GET request to the URL
response = requests.get(static_url)
# Check if the request was successful
if response.status_code == 200:
    # Parse the response text content using BeautifulSoup
    soup = BeautifulSoup(response.text, 'html.parser')
    print(soup.title)
else:
    print(f"Failed to retrieve the page. Status code: {response.status_code}")

<title>List of Falcon 9 and Falcon Heavy launches - Wikipedia</title>


### TASK 2: Extract all column/variable names from the HTML table header


In [29]:
html_tables = soup.find_all('table')
first_launch_table = html_tables[2]
column_names = []
# Extract the rows from the table (excluding the header)
rows = first_launch_table.find_all('tr')[1:]

# Iterate through each row and extract the relevant data
for row in rows:
    # Find all the table data cells in the row
    table_cells = row.find_all('td')
    
    # Extract data for specific columns (modify the index as per your column positions)
    if len(table_cells) >= 4:  # Assuming you have at least 4 columns in each row
        date_time_data = date_time(table_cells[0])  # Date and Time from the first column
        booster_version_data = booster_version(table_cells[1])  # Booster Version from the second column
        landing_status_data = landing_status(table_cells[2])  # Landing Status from the third column
        mass_data = get_mass(table_cells[3])  # Mass from the fourth column
        
        # You can now do something with this data, such as printing or storing it
        print(f"Date and Time: {date_time_data}, Booster Version: {booster_version_data}, "
              f"Landing Status: {landing_status_data}, Mass: {mass_data}")

print(column_names)

Date and Time: ['4 June 2010,', '18:45'], Booster Version: F9 v1.07B0003.18, Landing Status: CCAFS, Mass: D
Date and Time: ['8 December 2010,', '15:43'], Booster Version: F9 v1.07B0004.18, Landing Status: CCAFS, Mass: D
Date and Time: ['22 May 2012,', '07:44'], Booster Version: F9 v1.07B0005.18, Landing Status: CCAFS, Mass: D
Date and Time: ['8 October 2012,', '00:35'], Booster Version: F9 v1.07B0006.18, Landing Status: CCAFS, Mass: S
Date and Time: ['Orbcomm-OG2', '['], Booster Version: 172 kg (379 lb)24, Landing Status: LEO, Mass: O
Date and Time: ['1 March 2013,', '15:10'], Booster Version: F9 v1.07B0007.18, Landing Status: CCAFS, Mass: S
Date and Time: ['29 September 2013,', '16:00'], Booster Version: F9 v1.17B10038, Landing Status: VAFB, Mass: C
Date and Time: ['3 December 2013,', '22:41'], Booster Version: , Landing Status: CCAFS, Mass: S
[]
